In [1]:
import os
import cv2
import math
import torch
import random
import numpy as np
import pandas as pd

import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import swin3d_t, Swin3D_T_Weights

# =========================
# BASIC SETUP
# =========================
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data"
VIDEO_ROOT = os.path.join(ROOT, "GenderClips")
CSV_PATH = os.path.join(ROOT, "Labels", "AllLabels.csv")
SAVE_PATH = "BEST_DAISEE_BINARY_FINAL.pth"

SEED = 42
BATCH = 2
ACCUM = 4
EPOCHS = 15
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_CLASSES = 2
NUM_FRAMES = 16
IMG = 160
THRESHOLDS = np.arange(0.30, 0.81, 0.05)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


# =========================
# PATH HELPERS
# =========================
def find_video_path(clip_id: str):
    clip_id = str(clip_id).strip()

    male = os.path.join(VIDEO_ROOT, "Male", clip_id + ".avi")
    female = os.path.join(VIDEO_ROOT, "Female", clip_id + ".avi")

    if os.path.exists(male):
        return male
    if os.path.exists(female):
        return female
    return None


# =========================
# DATA PREP
# =========================
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
df["ClipID"] = df["ClipID"].astype(str).str.strip()

# Binary label
# 집중 = Engagement >= 2
# 비집중 = Engagement 0 또는 1
df["label"] = (df["Engagement"] >= 2).astype(int)

# 실제 파일 경로 매핑
df["video_path"] = df["ClipID"].apply(find_video_path)

total_rows = len(df)
missing_rows = df["video_path"].isna().sum()

print(f"전체 라벨 수: {total_rows}")
print(f"영상 못 찾은 수: {missing_rows}")

# 없는 영상 제거
df = df[df["video_path"].notna()].reset_index(drop=True)

print("실제 학습 가능 수:", len(df))
print("라벨 분포:")
print(df["label"].value_counts())

# stratify split
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label"],
    random_state=SEED
)

print("train label 분포:")
print(train_df["label"].value_counts())
print("val label 분포:")
print(val_df["label"].value_counts())


# =========================
# DATASET
# =========================
class DaiseeDataset(Dataset):
    def __init__(self, df: pd.DataFrame, train: bool = True):
        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def _sample_frame_indices(self, total_frames: int):
        if total_frames <= 0:
            return [0] * NUM_FRAMES

        if total_frames < NUM_FRAMES:
            indices = list(range(total_frames))
            while len(indices) < NUM_FRAMES:
                indices.append(indices[-1])
            return indices

        if self.train:
            # train: 구간별 랜덤 샘플링
            segments = np.linspace(0, total_frames, NUM_FRAMES + 1, dtype=int)
            indices = []
            for i in range(NUM_FRAMES):
                start = segments[i]
                end = max(segments[i + 1] - 1, start)
                idx = random.randint(start, end)
                indices.append(idx)
            return indices

        # val: 균등 샘플링
        return np.linspace(0, total_frames - 1, NUM_FRAMES, dtype=int).tolist()

    def load_video(self, path: str):
        if path is None or not os.path.exists(path):
            return torch.zeros(3, NUM_FRAMES, IMG, IMG, dtype=torch.float32)

        cap = cv2.VideoCapture(path)
        if not cap.isOpened():
            return torch.zeros(3, NUM_FRAMES, IMG, IMG, dtype=torch.float32)

        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = self._sample_frame_indices(total)

        frames = []
        for frame_idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()

            if not ret or frame is None:
                if len(frames) > 0:
                    frames.append(frames[-1].copy())
                else:
                    frames.append(np.zeros((IMG, IMG, 3), dtype=np.uint8))
                continue

            frame = cv2.resize(frame, (IMG, IMG), interpolation=cv2.INTER_AREA)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            if self.train and random.random() < 0.5:
                frame = cv2.flip(frame, 1)

            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            return torch.zeros(3, NUM_FRAMES, IMG, IMG, dtype=torch.float32)

        while len(frames) < NUM_FRAMES:
            frames.append(frames[-1].copy())

        frames = np.stack(frames[:NUM_FRAMES]).astype(np.float32) / 255.0
        frames = torch.from_numpy(frames).permute(3, 0, 1, 2).contiguous()

        return frames

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video = self.load_video(row["video_path"])
        label = int(row["label"])
        return video, label


train_dataset = DaiseeDataset(train_df, train=True)
val_dataset = DaiseeDataset(val_df, train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False
)


# =========================
# CLASS WEIGHT
# =========================
train_labels = train_df["label"].values
class_counts = np.bincount(train_labels, minlength=2)
class_weights = len(train_labels) / (2.0 * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

print("class_counts:", class_counts)
print("class_weights:", class_weights)


# =========================
# MODEL
# =========================
model = swin3d_t(weights=Swin3D_T_Weights.DEFAULT)

in_features = model.head.in_features
model.head = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, NUM_CLASSES)
)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.amp.GradScaler("cuda" if torch.cuda.is_available() else "cpu")


# =========================
# VALIDATION THRESHOLD SEARCH
# =========================
def evaluate_with_best_threshold(model, loader):
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for video, label in loader:
            video = video.to(DEVICE, non_blocking=True)
            video = video.contiguous(memory_format=torch.channels_last_3d)

            with torch.amp.autocast("cuda" if torch.cuda.is_available() else "cpu"):
                logits = model(video)
                probs = torch.softmax(logits, dim=1)[:, 1]

            all_probs.extend(probs.detach().cpu().numpy().tolist())
            all_labels.extend(label.numpy().tolist())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    best_threshold = 0.5
    best_f1 = -1
    best_acc = 0
    best_precision = 0
    best_recall = 0

    for th in THRESHOLDS:
        preds = (all_probs >= th).astype(int)

        f1 = f1_score(all_labels, preds, zero_division=0)
        acc = accuracy_score(all_labels, preds)
        precision = precision_score(all_labels, preds, zero_division=0)
        recall = recall_score(all_labels, preds, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = th
            best_acc = acc
            best_precision = precision
            best_recall = recall

    return {
        "f1": best_f1,
        "acc": best_acc,
        "precision": best_precision,
        "recall": best_recall,
        "threshold": best_threshold,
    }


# =========================
# TRAIN
# =========================
best_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for step, (video, label) in enumerate(pbar):
        video = video.to(DEVICE, non_blocking=True)
        video = video.contiguous(memory_format=torch.channels_last_3d)
        label = label.to(DEVICE, non_blocking=True)

        with torch.amp.autocast("cuda" if torch.cuda.is_available() else "cpu"):
            logits = model(video)
            loss = criterion(logits, label)
            loss = loss / ACCUM

        scaler.scale(loss).backward()

        should_step = ((step + 1) % ACCUM == 0) or ((step + 1) == len(train_loader))
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * ACCUM
        pbar.set_postfix(loss=f"{total_loss / (step + 1):.4f}")

    scheduler.step()

    metrics = evaluate_with_best_threshold(model, val_loader)

    print(
        f"[Epoch {epoch+1}] "
        f"Loss={total_loss / len(train_loader):.4f} "
        f"F1={metrics['f1']:.4f} "
        f"ACC={metrics['acc']:.4f} "
        f"P={metrics['precision']:.4f} "
        f"R={metrics['recall']:.4f} "
        f"TH={metrics['threshold']:.2f}"
    )

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "best_f1": best_f1,
                "best_threshold": metrics["threshold"],
                "img_size": IMG,
                "num_frames": NUM_FRAMES,
            },
            SAVE_PATH
        )
        print("BEST MODEL SAVED")

GPU: True
NVIDIA GeForce RTX 4060 Laptop GPU
전체 라벨 수: 8925
영상 못 찾은 수: 8925
실제 학습 가능 수: 0
라벨 분포:
Series([], Name: count, dtype: int64)


ValueError: With n_samples=0, test_size=0.15 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [14]:
!pip install kaggle -q
!pip install kagglehub -q

In [15]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olgaparfenova/daisee")

print("Path to dataset files:", path)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 14.3G/14.3G [3:32:24<00:00, 1.20MB/s]

Extracting files...


Path to dataset files: C:\Users\msi\.cache\kagglehub\datasets\olgaparfenova\daisee\versions\1


In [1]:
import os
import pandas as pd
import cv2
from tqdm import tqdm

# ==========================================
# 실제 환경에 맞춘 경로 설정 (주인님의 캡처 화면 참고)
# ==========================================
DATA_DIR = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DataSet" 
CSV_PATH = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\Labels\TrainLabels.csv"
SAVE_DIR = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\ExtractedFrames"

def extract_frames():
    print("주인님, 강화된 프레임 추출 로직을 시작합니다.")
    os.makedirs(SAVE_DIR, exist_ok=True)
    
    try:
        labels_df = pd.read_csv(CSV_PATH)
    except FileNotFoundError:
        print(f"CSV 파일을 찾을 수 없습니다: {CSV_PATH}")
        return

    not_found_count = 0

    for idx, row in tqdm(labels_df.iterrows(), total=len(labels_df), desc="프레임 추출 중"):
        clip_id = row['ClipID']
        clip_name = clip_id.split('.')[0]
        subject_id = clip_name[:6]

        # 💡 핵심: DAiSEE 데이터셋의 다양한 경로와 확장자 변수를 모두 체크합니다.
        possible_paths = [
            os.path.join(DATA_DIR, subject_id, clip_name, clip_id),                               # 기존 경로 (.avi)
            os.path.join(DATA_DIR, subject_id, clip_name, f"{clip_name}.mp4"),                    # 확장자가 mp4인 경우
            os.path.join(DATA_DIR, "Train", subject_id, clip_name, clip_id),                      # Train 폴더가 중간에 있는 경우
            os.path.join(DATA_DIR, "Train", subject_id, clip_name, f"{clip_name}.mp4")            # Train 폴더 + mp4인 경우
        ]

        video_path = None
        for path in possible_paths:
            if os.path.exists(path):
                video_path = path
                break

        # 파일을 결국 찾지 못한 경우
        if video_path is None:
            if not_found_count < 3: # 콘솔 도배 방지를 위해 처음 3개만 상세 출력
                print(f"\n⚠️ 파일을 찾을 수 없습니다. (예상 경로: {possible_paths[0]})")
            not_found_count += 1
            continue

        save_path = os.path.join(SAVE_DIR, f"{clip_name}.jpg")
        
        # 이미 추출된 사진이 있다면 패스 (중간에 끊겨도 재개 가능)
        if os.path.exists(save_path):
            continue

        cap = cv2.VideoCapture(video_path)
        ret, frame = cap.read()
        cap.release()

        if ret:
            cv2.imwrite(save_path, frame)
        else:
            print(f"\n⚠️ 파일은 존재하지만 프레임을 읽지 못했습니다: {video_path}")

    if not_found_count > 0:
        print(f"\n총 {not_found_count}개의 동영상 파일을 찾지 못했습니다. DAiSEE 데이터셋 폴더 구조를 한 번 더 확인해 보셔야 할 것 같습니다.")
    else:
        print("\n모든 프레임 추출이 완벽하게 끝났습니다!")

if __name__ == '__main__':
    extract_frames()

주인님, 강화된 프레임 추출 로직을 시작합니다.


프레임 추출 중: 100%|████████████████████████████████████████████████████████████████████████████████████| 5358/5358 [00:52<00:00, 101.42it/s]


모든 프레임 추출이 완벽하게 끝났습니다!


In [ ]:
# ==========================================
# 2. 데이터셋 클래스 (이미지 로드 방식으로 최적화)
# ==========================================
class DaiseeBinaryDataset(Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        if not os.path.exists(csv_file):
            raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다: {csv_file}")
            
        self.labels_df = pd.read_csv(csv_file)
        self.image_dir = image_dir # 추출된 이미지가 있는 폴더 경로 (SAVE_DIR)
        self.transform = transform
        
        self.labels_df['Focused'] = self.labels_df['Engagement'].apply(lambda x: 1 if x >= 2 else 0)

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        clip_id = self.labels_df.iloc[idx]['ClipID']
        clip_name = clip_id.split('.')[0] 
        
        # 미리 추출해둔 이미지 경로 사용
        img_path = os.path.join(self.image_dir, f"{clip_name}.jpg")
        
        try:
            # PIL Image로 바로 로드 (cv2 비디오 로드보다 압도적으로 빠름)
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            # 파일이 없을 경우 더미 이미지 반환
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        label = self.labels_df.iloc[idx]['Focused']
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

# ------------------------------------------
# ※ 학습 루프(train 함수) 내부의 데이터로더 호출 부분을 아래와 같이 수정해야 합니다.
# dataset = DaiseeBinaryDataset(csv_file=CSV_PATH, image_dir="./DAiSEE/ExtractedFrames", transform=transform)
# dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4) # 이제 일꾼(workers)을 늘려도 안전합니다.
# ------------------------------------------